# Biomarker Analysis Pipeline (v2)

Line-matched ICI vs never-ICI cohorts with two propensity score models.

### Matching schemes
- **1:1** — one control per ICI case, matched on (cancer_type, line_category)
- **1:k** — up to 3 controls per case

### Propensity score models
- **embeddings_only** — LR on text embeddings
- **all_covariates** — LR on embeddings + demographics + cancer type + panel version + line

### Analysis tracks
- **Track 1** — ICI-only, generalizability-weighted: `S(t) ~ base_vars + line_dummies + marker`
- **Track 2** — Full cohort, IPTW-weighted: `S(t) ~ base_vars + line_dummies + marker + ICI + marker×ICI`

### Stages
1. **Data regeneration** — `generate_all_non_text_covariates.py` (SV/Fusion fix)
2. **Line-matched cohorts** — `build_line_matched_cohort.py`
3. **Propensity scores** — `ICI_LRs.py --matching {1to1,1tok}`
4. **IPTW datasets** — `generate_IPTW_df.py --matching {1to1,1tok} --ps_model {embeddings_only,all_covariates}`
5. **Cox models** — `run_IPTW_analysis.py --matching {1to1,1tok} --ps_model {embeddings_only,all_covariates}`

In [ ]:
import subprocess
import sys
import os

SCRIPT_DIR = os.path.dirname(os.path.abspath('__file__'))
MATCHING_SCHEMES = ['1to1', '1tok']
PS_MODELS = ['embeddings_only', 'all_covariates']

def run_and_stream(label, cmd):
    """Run a command and stream its output inline."""
    print(f"\n--- {label} ---")
    result = subprocess.run(cmd, cwd=SCRIPT_DIR,
                            stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                            universal_newlines=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print(f"FAILED (exit code {result.returncode})")
        if result.stderr:
            print(result.stderr)
        raise RuntimeError(f"{label} failed")
    print(f"Done: {label}")

## Stage 1: Data Regeneration (SV/Fusion fix)

Re-run `generate_all_non_text_covariates.py` to ensure `complete_somatic_data_df.csv` includes correct SV/Fusion columns.

In [ ]:
run_and_stream('Data regeneration',
               [sys.executable, '../data_preprocessing/generate_all_non_text_covariates.py'])

## Stage 2: Line-Matched Cohort Construction

Build 1:1 and 1:k matched cohorts on (cancer_type, line_category).

In [ ]:
run_and_stream('Line-matched cohorts',
               [sys.executable, 'build_line_matched_cohort.py'])

## Stage 3: Propensity Score Generation

Train embeddings-only and all-covariates LR propensity models for each matching scheme.

In [ ]:
for matching in MATCHING_SCHEMES:
    run_and_stream(f'Propensity scores ({matching})',
                   [sys.executable, 'ICI_LRs.py', '--matching', matching])

## Stage 4: IPTW Dataset Generation

Build IPTW datasets for each {matching, ps_model} combination.

In [ ]:
for matching in MATCHING_SCHEMES:
    for ps_model in PS_MODELS:
        run_and_stream(f'IPTW dataset ({matching}, {ps_model})',
                       [sys.executable, 'generate_IPTW_df.py',
                        '--matching', matching, '--ps_model', ps_model])

## Stage 5: Cox Model Analysis

Run Track 1 (ICI-only, generalizability-weighted) and Track 2 (full-cohort interaction) for each specification.

In [ ]:
for matching in MATCHING_SCHEMES:
    for ps_model in PS_MODELS:
        run_and_stream(f'Cox models ({matching}, {ps_model})',
                       [sys.executable, 'run_IPTW_analysis.py',
                        '--matching', matching, '--ps_model', ps_model])

## Stage 6: Compile Results

Aggregate significant hits across all specifications for comparison.

In [ ]:
import re
import pandas as pd

OUTPUT_PATH = '/data/gusev/USERS/jpconnor/data/clinical_text_embedding_project/biomarker_analysis/'
COMPILED_PATH = os.path.join(OUTPUT_PATH, 'compiled_results/')
os.makedirs(COMPILED_PATH, exist_ok=True)

# Discover cancer types from result filenames
cancer_types = set()
for matching in MATCHING_SCHEMES:
    for ps_model in PS_MODELS:
        spec = f'{matching}_{ps_model}'
        run_path = os.path.join(OUTPUT_PATH, f'IPTW_runs_{spec}/')
        if not os.path.isdir(run_path):
            continue
        for fname in os.listdir(run_path):
            m = re.match(r'(.+)_track[12]_', fname)
            if m:
                cancer_types.add(m.group(1))
cancer_types = sorted(cancer_types)
print(f"Discovered cancer types: {cancer_types}")

# --- Track 2: compile interaction hits ---
track2_rows = []
for matching in MATCHING_SCHEMES:
    for ps_model in PS_MODELS:
        spec = f'{matching}_{ps_model}'
        run_path = os.path.join(OUTPUT_PATH, f'IPTW_runs_{spec}/')
        for ct in cancer_types:
            for weight_type in ['ATE', 'ATT', 'noIPTW']:
                fname = os.path.join(run_path, f'{ct}_track2_{weight_type}_interaction.csv')
                if not os.path.exists(fname):
                    continue
                df = pd.read_csv(fname)
                if 'significant_predictive' in df.columns:
                    hits = df.loc[df['significant_predictive']].copy()
                    hits['matching'] = matching
                    hits['ps_model'] = ps_model
                    hits['weight_type'] = weight_type
                    hits['cancer_type'] = ct
                    track2_rows.append(hits)

if track2_rows:
    track2_compiled = pd.concat(track2_rows, ignore_index=True)
    track2_compiled.to_csv(os.path.join(COMPILED_PATH, 'track2_all_significant_hits.csv'), index=False)
    print(f"Track 2: {len(track2_compiled)} significant hits across all specs")
    print(track2_compiled.groupby(['matching', 'ps_model', 'weight_type', 'cancer_type']).size())
else:
    print("Track 2: no significant hits found")

# --- Track 1: compile ICI-only hits ---
track1_rows = []
for matching in MATCHING_SCHEMES:
    for ps_model in PS_MODELS:
        spec = f'{matching}_{ps_model}'
        run_path = os.path.join(OUTPUT_PATH, f'IPTW_runs_{spec}/')
        for ct in cancer_types:
            for weight_type in ['weighted', 'unweighted']:
                fname = os.path.join(run_path, f'{ct}_track1_{weight_type}_ICI_only.csv')
                if not os.path.exists(fname):
                    continue
                df = pd.read_csv(fname)
                if 'significant_marker' in df.columns:
                    hits = df.loc[df['significant_marker']].copy()
                    hits['matching'] = matching
                    hits['ps_model'] = ps_model
                    hits['weight_type'] = weight_type
                    hits['cancer_type'] = ct
                    track1_rows.append(hits)

if track1_rows:
    track1_compiled = pd.concat(track1_rows, ignore_index=True)
    track1_compiled.to_csv(os.path.join(COMPILED_PATH, 'track1_all_significant_hits.csv'), index=False)
    print(f"\nTrack 1: {len(track1_compiled)} significant hits across all specs")
    print(track1_compiled.groupby(['matching', 'ps_model', 'weight_type', 'cancer_type']).size())
else:
    print("\nTrack 1: no significant hits found")